# 코드잇 스프린트 미션16: 이미지 모델 밴치마크 사이트 개발(추론 및 양자화)
---
여러분들은 지금까지 AI 스프린트 미션들을 수행하며 다양한 모델들을 학습 및 구현해보셨습니다. 
이번 미션에서는 그 모델들을 다시 가져와서, 여러 형태의 포맷으로 모델을 변환하여 저장해보는 실습을 해봅시다.




## 가이드라인
1. 이전 미션에서 다루었던 모델들 중 하나 이상을 자유롭게 선택하여 모델 학습을 진행합니다.
2. 아래의 3가지 타입의 모델로 변환하여 저장해봅시다.
    - `.pth` (PyTorch 기본 저장 형식)
    - `.pth` (양자화 된 버전)
    - `.onnx` (ONNX 형식)

## 데이터셋

`mnist data set`

- **데이터 구성**:
    - **학습**: 60,000장
    - **테스트**: 10,000장
    - **크기**: 28×28 grayscale
    - **클래스**: 0-9 숫자

**용량**: ~12MB (매우 작음)

## 사용 모델

`ViT`

# 학습 코드

## 1. 라이브러리

In [49]:
%pip install -r requirements.txt

   ---------------------------------------- 0.0/683.0 kB ? eta -:--:--
   - ------------------------------------- 30.7/683.0 kB 660.6 kB/s eta 0:00:01
   ----------------- ---------------------- 307.2/683.0 kB 3.8 MB/s eta 0:00:01
   ---------------------------------------  675.8/683.0 kB 5.3 MB/s eta 0:00:01
   ---------------------------------------- 683.0/683.0 kB 4.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/129.3 kB ? eta -:--:--
   ---------------------------------------- 129.3/129.3 kB 7.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: c:\Users\hambu\.pyenv\pyenv-win\versions\3.12.3\python.exe -m pip install --upgrade pip


In [50]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import time
import torch.ao.quantization as quantization
import os
import onnx
import onnxruntime as ort
import numpy as np
from PIL import Image

# 모델 클래스
from model import VisionTransformer, Config

In [14]:
class EvalConfig:
    """평가 설정"""
    data_root = './data'
    batch_size = 64
    num_workers = 2
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

## 2. 데이터 로드

In [15]:
def load_test_data(config):
    """MNIST 테스트 데이터셋 로드"""
    print("=" * 60)
    print("데이터 로드 중...")
    print("=" * 60)
    
    test_dataset = datasets.MNIST(
        root=config.data_root,
        train=False,
        download=True,
        transform=transforms.ToTensor()
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=config.batch_size,
        shuffle=False,
        num_workers=config.num_workers,
        pin_memory=torch.cuda.is_available()
    )
    
    print(f"✅ 테스트 데이터: {len(test_dataset)}개")
    print(f"✅ 배치 수: {len(test_loader)}")
    print(f"✅ 배치 크기: {config.batch_size}")
    
    return test_loader

## 3. 모델 정의 및 평가

In [38]:
def evaluate_model(model, test_loader, device):
    """모델 평가"""
    print("\n" + "=" * 60)
    print("모델 평가 시작")
    print("=" * 60)
    
    model.eval()
    criterion = nn.CrossEntropyLoss()
    
    running_loss = 0.0
    correct = 0
    total = 0
    
    start_time = time.time()
    
    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(test_loader):
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            if (batch_idx + 1) % 50 == 0:
                print(f"  [{batch_idx+1}/{len(test_loader)}] 처리 중...")
    
    eval_time = time.time() - start_time
    avg_loss = running_loss / len(test_loader)
    accuracy = 100. * correct / total
    
    print("\n" + "=" * 60)
    print("평가 완료!")
    print("=" * 60)
    print(f"Loss: {avg_loss:.4f}")
    print(f"Accuracy: {accuracy:.2f}%")
    print(f"정답: {correct}/{total}")
    print(f"소요 시간: {eval_time:.1f}s")
    
    return {
        'loss': avg_loss,
        'accuracy': accuracy,
        'correct': correct,
        'total': total,
        'time': eval_time
    }

## 4. 모델 양자화

## 4. 모델 양자화
---

웹이나 CPU환경에서도 모델을 사용할 수 있도록 양자화를 진행할 예정이다.

### Dynamic Quantization
가장 간단하고 안전한 방법

특징:
- Linear 레이어만 INT8로 변환
- Inference 시점에 동적 양자화
- 캘리브레이션 불필요
- 정확도 손실 최소 (0.5% 이내)

적용 대상:              
✅ Linear (QKV projection, MLP, Head)           
❌ LayerNorm, GELU, Softmax (FP32 유지)         
❌ Attention 연산 (정확도 중요)

장점:
+ 구현 간단 (코드 3줄)
+ ViT에 최적 (Linear 레이어 많음)
+ 웹 배포에 적합
+ 모델 크기 ~60% 감소

단점:
- 추론 속도 향상 제한적 (CPU 20-30%)

### Static Quantization
더 나은 성능, 복잡도 증가

특징:
- Conv, Linear 모두 INT8
- 캘리브레이션 데이터셋 필요 (100-1000장)
- Activation도 양자화
- 정확도 손실 약간 증가 (1-2%)

적용 대상:              
✅ Conv2d (Patch Embedding)         
✅ Linear (전체)            
✅ Activation (ReLU 등)             
❌ LayerNorm, Softmax (민감)

장점:
+ 추론 속도 크게 향상 (CPU 2-3배)
+ 모델 크기 ~75% 감소
+ 메모리 사용량 감소

단점:
- 구현 복잡 (캘리브레이션 필요)
- 정확도 하락 위험
- 디버깅 어려움

### QAT (Quantization Aware Training)
최고 성능, 재학습 필요

특징:
- 학습 중에 양자화 시뮬레이션
- Fake quantization 사용
- 정확도 거의 유지 (0.1-0.5%)
- 재학습 시간 필요

적용:
- Stage 2 결과가 불만족스러울 때만

장점:
+ 정확도 손실 최소
+ Static Quantization + 정확도 복구

단점:
- 재학습 필요 (시간 많이 소요)
- 과제에는 과도함


### 최종 선택 전략: Dynamic Quantization

#### 이유

1. ViT 구조에 최적

    - ViT는 Linear 레이어가 전체 파라미터의 90% 이상
    - Attention은 FP32 유지 (정확도 보존)


2. 웹 배포에 적합

    - ONNX 변환 호환성 좋음
    - onnxruntime-web 지원
    - 모델 크기 크게 감소


3. 구현 간단

    - 코드 몇 줄로 완성
    - 디버깅 쉬움
    - 안정적


4. 과제 요구사항 충족

    - `.pth` (양자화) 저장 ✅
    - ONNX 변환 가능 ✅
    - TypeScript 웹에서 동작 ✅

In [39]:
def quantize_model(model):
    """Dynamic Quantization 적용"""
    print("\n" + "=" * 60)
    print("모델 양자화 중...")
    print("=" * 60)
    
    model.eval()
    model_quantized = quantization.quantize_dynamic(
        model,
        qconfig_spec={torch.nn.Linear},
        dtype=torch.qint8
    )
    
    print("✅ 양자화 완료 (Linear 레이어 → INT8)")
    
    return model_quantized

In [40]:
def get_model_size(model, filename='temp_model.pth'):
    """모델 크기 계산 (MB)"""
    torch.save(model.state_dict(), filename)
    size_mb = os.path.getsize(filename) / (1024 * 1024)
    os.remove(filename)
    return size_mb

In [41]:
def compare_models(model_original, model_quantized, test_loader, device):
    """두 모델 비교"""
    print("\n" + "=" * 60)
    print("모델 비교 분석")
    print("=" * 60)
    
    # 1. 정확도 평가
    print("\n📊 정확도 평가 중...")
    results_orig = evaluate_model(model_original, test_loader, device)
    
    print("\n📊 양자화 모델 평가 중...")
    results_quant = evaluate_model(model_quantized, test_loader, device)
    
    # 2. 모델 크기 비교
    print("\n💾 모델 크기 계산 중...")
    size_orig = get_model_size(model_original, 'temp_orig.pth')
    size_quant = get_model_size(model_quantized, 'temp_quant.pth')
    
    # 3. 결과 출력
    print("\n" + "=" * 60)
    print("비교 결과")
    print("=" * 60)
    
    print("\n📈 정확도:")
    print(f"  원본 모델:     {results_orig['accuracy']:.2f}%")
    print(f"  양자화 모델:   {results_quant['accuracy']:.2f}%")
    print(f"  정확도 차이:   {abs(results_orig['accuracy'] - results_quant['accuracy']):.2f}%")
    
    print("\n💾 모델 크기:")
    print(f"  원본 모델:     {size_orig:.2f} MB")
    print(f"  양자화 모델:   {size_quant:.2f} MB")
    print(f"  크기 감소:     {((size_orig - size_quant) / size_orig * 100):.1f}%")
    
    print("\n⏱️  추론 시간:")
    print(f"  원본 모델:     {results_orig['time']:.2f}s")
    print(f"  양자화 모델:   {results_quant['time']:.2f}s")
    print(f"  속도 향상:     {((results_orig['time'] - results_quant['time']) / results_orig['time'] * 100):.1f}%")
    
    return {
        'original': results_orig,
        'quantized': results_quant,
        'size_reduction': (size_orig - size_quant) / size_orig * 100,
        'speed_improvement': (results_orig['time'] - results_quant['time']) / results_orig['time'] * 100
    }

## 5. 웹 배포용 ONNX 레벨 양자화

In [ ]:
def convert_to_onnx(model, save_path, input_shape=(1, 1, 28, 28), opset_version=17):
    """PyTorch 모델을 ONNX로 변환"""
    print("\n" + "=" * 60)
    print(f"ONNX 변환 중: {save_path}")
    print("=" * 60)
    
    model.eval()
    model.to('cpu')
    
    # 더미 입력 생성
    dummy_input = torch.randn(input_shape)
    
    # ONNX 변환
    torch.onnx.export(
        model,
        dummy_input,
        save_path,
        export_params=True,
        opset_version=opset_version,
        do_constant_folding=True,
        input_names=['input'],
        output_names=['output'],
        dynamic_axes={
            'input': {0: 'batch_size'},
            'output': {0: 'batch_size'}
        }
    )
    
    print(f"✅ ONNX 변환 완료: {save_path}")

In [43]:
def verify_onnx(onnx_path):
    """ONNX 모델 검증"""
    print("\n" + "=" * 60)
    print("ONNX 모델 검증 중...")
    print("=" * 60)
    
    try:
        # 모델 로드
        onnx_model = onnx.load(onnx_path)
        
        # 유효성 검사
        onnx.checker.check_model(onnx_model)
        
        print("✅ ONNX 모델이 유효합니다!")
        
        # 모델 정보 출력
        print(f"\n📊 모델 정보:")
        print(f"  IR Version: {onnx_model.ir_version}")
        print(f"  Opset Version: {onnx_model.opset_import[0].version}")
        
        return True
        
    except Exception as e:
        print(f"❌ ONNX 검증 실패: {e}")
        return False


## 메인 실행

In [58]:
def main():
    """양자화 및 평가 파이프라인"""
    eval_config = EvalConfig()
    model_config = Config()
    
    print("=" * 60)
    print("ViT - Quantization & ONNX Conversion")
    print("=" * 60)
    print(f"디바이스: {eval_config.device}")
    print(f"배치 크기: {eval_config.batch_size}")
    
    model_path = input("\n원본 모델 경로: ")
    
    # 1. 데이터 로드
    test_loader = load_test_data(eval_config)
    
    # 2. 원본 모델 로드
    print("\n" + "=" * 60)
    print("원본 모델 로드 중...")
    print("=" * 60)
    
    model_original = VisionTransformer(model_config).to(eval_config.device)
    
    try:
        weights = torch.load(model_path, map_location=eval_config.device)
        model_original.load_state_dict(weights)
        print(f"✅ 모델 로드 완료: {model_path}")
    except FileNotFoundError:
        print(f"❌ 파일을 찾을 수 없습니다: {model_path}")
        return
    except Exception as e:
        print(f"❌ 모델 로드 실패: {e}")
        return
    
    # 3. 양자화
    model_quantized = quantize_model(model_original)
    
    # 4. 비교 분석
    comparison = compare_models(
        model_original,
        model_quantized,
        test_loader,
        eval_config.device
    )
    
    # 5. 양자화 모델 저장
    print("\n" + "=" * 60)
    print("양자화 모델 저장 중...")
    print("=" * 60)
    
    os.makedirs('./models', exist_ok=True)
    
    base_name = os.path.splitext(os.path.basename(model_path))[0]
    # 파일명에서 특수문자 제거 (안전)
    base_name = base_name.replace('(', '_').replace(')', '_').replace('=', '_')
    
    quantized_path = f'./models/{base_name}_quantized.pth'
    torch.save(model_quantized.state_dict(), quantized_path)
    print(f"✅ 양자화 모델 저장 완료: {quantized_path}")
    
    # 6. ONNX 변환 (수정!)
    print("\n" + "=" * 60)
    print("ONNX 변환 중...")
    print("=" * 60)
    
    onnx_path = f'./models/{base_name}.onnx'
    
    # 기존 모델을 CPU로 이동 (중복 생성 방지)
    model_original.to('cpu')
    
    convert_to_onnx(
        model_original,
        onnx_path,
        input_shape=(1, 1, 28, 28),
        opset_version=17  # ← 수정!
    )
    
    # 7. ONNX 검증 (수정!)
    if not verify_onnx(onnx_path):
        print("❌ ONNX 검증 실패. 작업을 중단합니다.")
        return  # ← 추가!
    
    # 8. External Data 통합 (추가!)
    data_path = onnx_path + '.data'
    if os.path.exists(data_path):
        print("\n" + "=" * 60)
        print("External Data 감지 - 통합 중...")
        print("=" * 60)
        
        try:
            import onnx
            
            # 통합 전 파일 크기 확인
            original_onnx_size = os.path.getsize(onnx_path) / 1024 / 1024
            data_size = os.path.getsize(data_path) / 1024 / 1024
            
            print(f"  원본 .onnx: {original_onnx_size:.2f} MB")
            print(f"  .data: {data_size:.2f} MB")
            
            # 통합 수행
            model_onnx = onnx.load(onnx_path, load_external_data=True)
            merged_path = onnx_path.replace('.onnx', '_merged.onnx')
            onnx.save(model_onnx, merged_path, save_as_external_data=False)
            
            merged_size = os.path.getsize(merged_path) / 1024 / 1024
            print(f"  통합 파일: {merged_size:.2f} MB")
            
            # 원본 파일 제거
            print("\n🗑️  원본 파일 제거 중...")
            os.remove(onnx_path)
            print(f"  ✅ 삭제: {os.path.basename(onnx_path)}")
            
            os.remove(data_path)
            print(f"  ✅ 삭제: {os.path.basename(data_path)}")
            
            print(f"\n✅ 통합 완료: {os.path.basename(merged_path)}")
            onnx_path = merged_path  # 경로 업데이트
            
        except Exception as e:
            print(f"⚠️  통합 실패: {e}")
            print("분리된 파일을 그대로 유지합니다.")
    else:
        print("\n✅ 단일 ONNX 파일로 생성됨 (External Data 없음)")
    
    # 9. 타겟 이미지 테스트 (제거 또는 기본 n)
    # 제거!
    
    # 10. 최종 요약
    print("\n" + "=" * 60)
    print("✅ 모든 작업 완료!")
    print("=" * 60)
    print(f"📁 원본 모델:     {model_path}")
    print(f"📁 양자화 모델:   {quantized_path}")
    print(f"📁 ONNX 모델:     {onnx_path}")
    print(f"📊 크기 감소:     {comparison['size_reduction']:.1f}%")
    print(f"🎯 정확도 유지:   {comparison['quantized']['accuracy']:.2f}%")

In [59]:
if __name__ == "__main__":
    main()

ViT - Quantization & ONNX Conversion
디바이스: cpu
배치 크기: 64
데이터 로드 중...
✅ 테스트 데이터: 10000개
✅ 배치 수: 157
✅ 배치 크기: 64

원본 모델 로드 중...
✅ 모델 로드 완료: ./models/mission_16_ViT_v1(epoch=10).pth

모델 양자화 중...
✅ 양자화 완료 (Linear 레이어 → INT8)

모델 비교 분석

📊 정확도 평가 중...

모델 평가 시작


C:\Users\hambu\AppData\Local\Temp\ipykernel_27988\1315085143.py:8: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_quantized = quantization.quantize_dynamic(


  [50/157] 처리 중...
  [100/157] 처리 중...
  [150/157] 처리 중...

평가 완료!
Loss: 0.1906
Accuracy: 93.96%
정답: 9396/10000
소요 시간: 13.9s

📊 양자화 모델 평가 중...

모델 평가 시작
  [50/157] 처리 중...
  [100/157] 처리 중...
  [150/157] 처리 중...


C:\Users\hambu\AppData\Local\Temp\ipykernel_27988\1410711907.py:14: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W1206 09:58:07.723000 27988 site-packages\torch\onnx\_internal\exporter\_compat.py:114] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features



평가 완료!
Loss: 0.1913
Accuracy: 93.97%
정답: 9397/10000
소요 시간: 11.7s

💾 모델 크기 계산 중...

비교 결과

📈 정확도:
  원본 모델:     93.96%
  양자화 모델:   93.97%
  정확도 차이:   0.01%

💾 모델 크기:
  원본 모델:     4.60 MB
  양자화 모델:   1.24 MB
  크기 감소:     73.0%

⏱️  추론 시간:
  원본 모델:     13.93s
  양자화 모델:   11.71s
  속도 향상:     16.0%

양자화 모델 저장 중...
✅ 양자화 모델 저장 완료: ./models/mission_16_ViT_v1_epoch_10__quantized.pth

ONNX 변환 중...

ONNX 변환 중: ./models/mission_16_ViT_v1_epoch_10_.onnx
[torch.onnx] Obtain model graph for `VisionTransformer([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `VisionTransformer([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 15 of general pattern rewrite rules.
✅ ONNX 변환 완료: ./models/mission_16_ViT_v1_epoch_10_.onnx

ONNX 모델 검증 중...
✅ ONNX 모델이 유효합니다!

📊 모델 정보:
  IR Version: 10
  Opset Version: 17

External Data 감지 - 통합 중...
  원본 .onnx: 0.03 MB
  .data: 4.58 MB
  통합 파일: 4.60 MB

🗑️  원본 파일 제거 중...
  ✅ 삭제: mission_16_ViT_v1_epoch_10_.onnx
  ✅ 삭제: mission_16_ViT_v1_epoch_10_.onnx.data

✅ 통합 완료: mission_16_ViT_v1_epoch_10__merged.onnx

✅ 모든 작업 완료!
📁 원본 모델:     ./models/mission_16_ViT_v1(epoch=10).pth
📁 양자화 모델:   ./models/mission_16_ViT_v1_epoch_10__quantized.pth
📁 ONNX 모델:     ./models/mission_16_ViT_v1_epoch_10__merged.onnx
📊 크기 감소:     73.0%
🎯 정확도 유지:   93.97%
